# Fareez Clinician-Rating Error Analysis

Distinguishes (a) **inherent chart-overreach risk** intrinsic to LLM behaviour and (b) **methodology-specific exacerbation** introduced by the synthetic patient-pairing design used for the Fareez OSCE evaluation. Built in response to reviewer critique #3.

Quick definitions, derived from clinician comment text in `fareez_clinician_ratings.csv`:

- **EHR-included error**: comment contains language like `EHR inclusion`, `imports EHR`, `mismatched EHR`, `comorbidity inclusion`, `extra info`, or similar — the model brought chart content into the note that wasn't discussed in the transcript.
- **EHR-correctly-ignored**: comment explicitly notes the model `correctly ignores`, `appropriately handles`, or `did not import` mismatched EHR — the desirable behaviour.
- **Fabrication**: comment mentions `fabricat`, `invented`, `hallucinat`, or `made up` — pure model fabrication independent of the chart.
- **Pairing-specific (gender mismatch)**: comment mentions `gender mismatch` or `mismatched patient profiles` — only possible because the synthetic OpenEMR patient was condition-matched but not patient-matched, an artifact of the methodology.

In [ ]:
import pandas as pd
import re
from pathlib import Path

df = pd.read_csv(Path('.').resolve() / 'fareez_clinician_ratings.csv')
df['comments_l'] = df['comments'].fillna('').str.lower()
print(f'Total ratings: {len(df)}')
print(f'Distinct summaries: {df.summary_id.nunique()}')
print(f'Raters: {sorted(df.rater.unique())}')
print(f'Models: {sorted(df.model.unique())}')

In [ ]:
# Classify each rating's comment
EHR_INCLUDED = re.compile(
    r'ehr (inclusion|comorbidity|fact|history|conditions?)|imports? ehr|'
    r'mismatched ehr (?!correctly|appropriately|but not)|extra info|'
    r'unmentioned (additions?|plan|meds?|comorbidit)|'
    r'(includes|adds|inserts)( |s )(unmentioned|chart|background)|'
    r'pulled? from (the )?(chart|ehr)|chart content not (in|from) (the )?(transcript|encounter)',
    re.IGNORECASE,
)
EHR_IGNORED = re.compile(
    r'(correctly|appropriately) ignor(es|ed) (the )?(mismatched )?ehr|'
    r'(correctly|appropriately) handle[ds]( the)? (mismatched )?(ehr|patient profile)|'
    r'did not (import|include) (mismatched )?ehr|'
    r'avoid(s|ed) ehr',
    re.IGNORECASE,
)
FABRICATION = re.compile(
    r'fabricat|invent(ed|s|ion)|hallucinat(ed|ion|es)|made.up|incorrect medication|'
    r'wrong (med|drug|dose)|invented (exam|finding|plan)',
    re.IGNORECASE,
)
GENDER_MISMATCH = re.compile(
    r'gender mismatch|mismatched patient (profile|record)|sex mismatch',
    re.IGNORECASE,
)

df['flag_ehr_included']    = df['comments_l'].str.contains(EHR_INCLUDED).astype(int)
df['flag_ehr_ignored']     = df['comments_l'].str.contains(EHR_IGNORED).astype(int)
df['flag_fabrication']     = df['comments_l'].str.contains(FABRICATION).astype(int)
df['flag_gender_mismatch'] = df['comments_l'].str.contains(GENDER_MISMATCH).astype(int)

summary = df[['flag_ehr_included','flag_ehr_ignored','flag_fabrication','flag_gender_mismatch']].sum()
print('Flag counts (out of 480 ratings):')
print(summary)
print()
print('Pct of ratings:')
print((summary / len(df) * 100).round(1).astype(str) + '%')

## Accuracy by error subgroup

Per the paper, the grand-mean accuracy was 2.55/5.0. The subgroups below isolate the contribution of the synthetic-pairing artefact (gender mismatch) and the EHR-overreach error class.

In [ ]:
def group_stat(mask, name):
    n = mask.sum()
    if n == 0:
        return {'group': name, 'n': 0, 'mean_acc': None, 'mean_overall': None}
    return {
        'group': name,
        'n': int(n),
        'mean_acc': round(df.loc[mask, 'accuracy'].mean(), 3),
        'mean_overall': round(df.loc[mask, 'overall_quality'].mean(), 3),
    }

rows = [
    group_stat(df['flag_ehr_included'].astype(bool),    'EHR INCLUDED (overreach)'),
    group_stat(df['flag_ehr_ignored'].astype(bool),     'EHR correctly IGNORED'),
    group_stat(df['flag_fabrication'].astype(bool),     'Fabrication'),
    group_stat(df['flag_gender_mismatch'].astype(bool), 'Pairing artefact (gender mismatch)'),
    group_stat(~df['flag_ehr_included'].astype(bool) & ~df['flag_fabrication'].astype(bool),
               'No EHR-overreach AND no fabrication'),
    group_stat(pd.Series([True]*len(df)), 'ALL ratings'),
]
out = pd.DataFrame(rows)
out

## Decomposition of the ~32% EHR-contamination figure quoted in the paper

The paper's Error Analysis says "approximately 32% of clinician comments" flagged EHR-not-in-transcript. We split this into:
- portion attributable to the **synthetic-pairing methodology** (visible primarily in `gender_mismatch`-flagged comments — these errors would not occur in a true patient-matched deployment)
- residual portion that is the **inherent chart-overreach failure mode** (would persist in any production deployment where the chart contains background context not raised in the visit)

In [ ]:
# Among EHR-included comments, how many also have gender-mismatch (pairing artefact)?
ei = df['flag_ehr_included'].astype(bool)
gm = df['flag_gender_mismatch'].astype(bool)
n_ei = int(ei.sum())
n_ei_gm = int((ei & gm).sum())
n_ei_no_gm = int((ei & ~gm).sum())
print(f'EHR-included flagged ratings:                {n_ei}  ({n_ei/len(df)*100:.1f}% of {len(df)})')
print(f'  also gender-mismatch (pairing artefact):   {n_ei_gm}  ({n_ei_gm/len(df)*100:.1f}%)')
print(f'  WITHOUT gender-mismatch (inherent risk):   {n_ei_no_gm}  ({n_ei_no_gm/len(df)*100:.1f}%)')
print()
print('Interpretation: of the EHR-overreach error class, the lower bound on the')
print('inherent (production-relevant) portion is the WITHOUT-gender-mismatch share.')
print('Comments that combine both could plausibly belong to either category and are')
print('not separately attributable from the comment text alone.')

In [ ]:
# Per-model breakdown
by_model = df.groupby('model').agg(
    n_ratings=('summary_id','count'),
    pct_ehr_included=('flag_ehr_included', lambda s: round(s.mean()*100, 1)),
    pct_ehr_ignored=('flag_ehr_ignored', lambda s: round(s.mean()*100, 1)),
    pct_fabrication=('flag_fabrication', lambda s: round(s.mean()*100, 1)),
    pct_gender_mismatch=('flag_gender_mismatch', lambda s: round(s.mean()*100, 1)),
    mean_accuracy=('accuracy','mean'),
)
by_model

## Accuracy upper bound under perfect EHR matching

The 6-case institutional evaluation (paper Table 2) used patient-matched EHR records (no synthetic pairing). If we treat **EHR-correctly-ignored** ratings on the Fareez set as a behavioural proxy for what the model does when the chart is appropriately bounded, the resulting subgroup accuracy is an estimate of the upper bound on accuracy that a perfectly patient-matched Fareez evaluation would have produced.

In [ ]:
ignored = df['flag_ehr_ignored'].astype(bool)
print(f'Subgroup: EHR correctly ignored  (n={int(ignored.sum())})')
print(f'  mean accuracy:        {df.loc[ignored, "accuracy"].mean():.2f}/5.0')
print(f'  mean completeness:    {df.loc[ignored, "completeness"].mean():.2f}/5.0')
print(f'  mean overall_quality: {df.loc[ignored, "overall_quality"].mean():.2f}/5.0')
print()
print('Compare grand-mean accuracy (all 480 ratings): 2.55/5.0')
print('Compare 6-case institutional evaluation accuracy upper bound: 3.75/5.0')

## Save consolidated subgroup table for the paper revision

In [ ]:
out.to_csv('error_subgroup_summary.csv', index=False)
by_model.to_csv('error_subgroup_by_model.csv')
print('wrote error_subgroup_summary.csv and error_subgroup_by_model.csv')